## !!! FIRST RUN ONLY

In [ ]:
# 0. CELL — Create venv, install packages, register as Jupyter kernel (First run only)

import platform
import subprocess
import sys
from pathlib import Path

env_name = "od_env"
python_exec = sys.executable
print(f"Detected OS: {platform.system()}")

# Create virtual environment if missing
if not Path(env_name).exists():
    print(f"Creating virtual environment: {env_name}")
    subprocess.run([python_exec, "-m", "venv", env_name], check=True)
else:
    print("[INFO] Virtual environment already exists.")

# Determine python binary in venv
if platform.system() == "Windows":
    venv_python = str(Path(env_name) / "Scripts" / "python.exe")
else:
    venv_python = str(Path(env_name) / "bin" / "python")

# Upgrade pip
print("[INFO] Upgrading pip...")
subprocess.run([venv_python, "-m", "pip", "install", "--upgrade", "pip"], check=True)

print("[INFO] Installing core dependencies (CUDA-capable PyTorch + Vision)...")
subprocess.run([
    venv_python, "-m", "pip", "install",
    "torch", "torchvision", "--extra-index-url",
    "https://download.pytorch.org/whl/cu121"
], check=True)

print("[INFO] Installing supporting libraries...")
subprocess.run([
    venv_python, "-m", "pip", "install",
    "ultralytics",
    "pycocotools",
    "albumentations",
    "tensorboard",
    "opencv-python",
    "tqdm"
], check=True)

print("[INFO] Installing missing required libs (YAML + Pandas + Matplotlib)...")
subprocess.run([venv_python, "-m", "pip", "install",
                "pyyaml", "pandas", "matplotlib"], check=True)

print("[INFO] Registering environment as a Jupyter kernel...")
subprocess.run([venv_python, "-m", "pip", "install", "ipykernel"], check=True)
subprocess.run([venv_python, "-m", "ipykernel", "install",
                "--name", env_name, "--display-name", "Python (od_env)"], check=True)

print("="*100)
print("[OK] Environment setup complete.")
print(">>> Restart Jupyter and select kernel: Python (od_env)")
print("="*100)

## Imports + CUDA check

In [ ]:
# 1. CELL — Environment & Library Setup

# Standard library
import os
import random
import numpy as np
from pathlib import Path
import yaml
import pandas as pd

# PyTorch core
import torch
import torchvision
from torchvision import transforms

# YOLO
try:
    from ultralytics import YOLO
except ImportError:
    raise ImportError(
        "[ERROR] Ultralytics is not installed. "
        "Run Cell 0 or pip install ultralytics"
    )

# COCO evaluation API
try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except ImportError:
    raise ImportError(
        "[ERROR] pycocotools is not installed. "
        "Run Cell 0 or pip install pycocotools"
    )

# Data augmentation library
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ImportError:
    raise ImportError(
        "[ERROR] Albumentations is not installed. "
        "Run Cell 0 or pip install albumentations"
    )

# Image I/O
try:
    import cv2
except ImportError:
    raise ImportError(
        "[ERROR] OpenCV is not installed. "
        "Run Cell 0 or pip install opencv-python"
    )

# Plotting
try:
    import matplotlib.pyplot as plt
except ImportError:
    raise ImportError(
        "[ERROR] Matplotlib is not installed. "
        "Run Cell 0 or pip install matplotlib"
    )

# Optional logging
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:
    print("[WARN] TensorBoard not found. Logging disabled.")

# GPU Check
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Active CUDA device:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

if not torch.cuda.is_available():
    raise EnvironmentError(
        "[ERROR] CUDA not found. "
        "This object detection pipeline requires an NVIDIA GPU."
    )

print("[INFO] Environment and libraries loaded successfully.")


## Global Config

In [ ]:
# 2. CELL — Global Configuration Settings

from types import SimpleNamespace
from pathlib import Path
from datetime import datetime
import multiprocessing

def attach_timestamp(name):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{name}_{stamp}"

# Auto-suggest sensible num_workers based on CPU count
DEFAULT_WORKERS = max(2, multiprocessing.cpu_count() // 2)

CONFIG = SimpleNamespace(

    # MODEL SELECTION
    model_type = "yolo",          # "yolo", "fasterrcnn", "detr"
    model_name = "yolov8n.pt",    # Only used for YOLO

    # DATA SETTINGS
    dataset_root = "/content/data",
    data_format = "yolo",         # "yolo" or "coco"
    class_names = [],             # Auto-detected in Cell 4
    num_classes = None,           # Must be set by Cell 4
    include_background = True,    # TorchVision/DETR only

    # TRAINING PARAMETERS
    epochs = 50,
    batch_size = 16,
    learning_rate = 0.001,
    img_size = 640,
    optimizer = "adamw",
    scheduler = "cosine",

    # CUDA Performance Flags
    amp = True,                   # Mixed precision ON (Tensor cores benefit)
    num_workers = DEFAULT_WORKERS,
    pin_memory = True,            # Faster host-to-device transfers
    random_seed = 42,

    # LOGGING / OUTPUT
    run_name = "experiment_001",
    use_tensorboard = True,
    base_out_dir = "./runs",
    log_dir = None,
    ckpt_dir = None,
    export_dir = None,

    # EXPORT OPTIONS
    export_onnx = True,
    export_trt = True,
    export_tflite = False,
    export_coreml = False
)

# Resolve output paths
root = Path(CONFIG.base_out_dir) / CONFIG.model_type / CONFIG.run_name

# Prevent accidental overwrite
if root.exists():
    print(f"[WARN] run '{CONFIG.run_name}' exists. Appending timestamp.")
    CONFIG.run_name = attach_timestamp(CONFIG.run_name)
    root = Path(CONFIG.base_out_dir) / CONFIG.model_type / CONFIG.run_name

CONFIG.log_dir    = str(root / "logs")
CONFIG.ckpt_dir   = str(root / "checkpoints")
CONFIG.export_dir = str(root / "exports")

for d in [CONFIG.log_dir, CONFIG.ckpt_dir, CONFIG.export_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Format validation
if CONFIG.model_type == "yolo" and CONFIG.data_format != "yolo":
    raise ValueError("[CONFIG ERROR] YOLO requires data_format='yolo'.")
if CONFIG.model_type != "yolo" and CONFIG.data_format != "coco":
    raise ValueError("[CONFIG ERROR] FasterRCNN/DETR require data_format='coco'.")

# Background flag forced off for YOLO
if CONFIG.model_type == "yolo":
    CONFIG.include_background = False

print(f"[INFO] Config OK. Run directory prepared: {root}")
print(f"[INFO] num_workers={CONFIG.num_workers}, pin_memory={CONFIG.pin_memory}, AMP={CONFIG.amp}")
print(CONFIG)


## Device & Seed Setup

In [ ]:
# 3. CELL — Device & Seed Setup

import torch
import random
import numpy as np

print("[INFO] Initializing CUDA device and reproducibility settings...")

# Select compute device (CUDA required in this project)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# Safety guard
if device.type != "cuda":
    raise EnvironmentError(
        "[ERROR] CUDA device not found. "
        "This training pipeline requires an NVIDIA GPU."
    )

# Set seeds (CPU + NumPy + Python)
torch.manual_seed(CONFIG.random_seed)
random.seed(CONFIG.random_seed)
np.random.seed(CONFIG.random_seed)

# Set CUDA RNG seeds (ALL GPUs if multiple available)
torch.cuda.manual_seed_all(CONFIG.random_seed)

# Performance vs Determinism trade-off:
# YOLO & Vision models benefit heavily from cuDNN autotuner when input size is fixed.
torch.backends.cudnn.benchmark = True     # Fast convolution tuning enabled
torch.backends.cudnn.deterministic = False  # Slight nondeterminism allowed for speed

print(f"[INFO] Seeds set. cuDNN benchmark=True, deterministic=False")
print("[INFO] Device and seed setup complete.")


## Dataset Validation & Class Extraction

In [ ]:
# 4. CELL — Dataset Validation + Class Discovery

import os
import yaml
from pathlib import Path
import json

root = Path(CONFIG.dataset_root)

train_images = root / "train" / "images"
train_labels = root / "train" / "labels"
val_images   = root / "val" / "images"
val_labels   = root / "val" / "labels"

# -----------------------------
# YOLO DATASET VALIDATION
# -----------------------------
if CONFIG.data_format == "yolo":
    for p in [train_images, train_labels, val_images, val_labels]:
        if not p.exists():
            raise FileNotFoundError(f"[ERROR] Missing folder: {p}")

    def validate_yolo_pairs(image_dir, label_dir):
        imgs = {f.stem for f in image_dir.glob("*") if f.suffix.lower() in [".jpg", ".png", ".jpeg"]}
        lbls = {f.stem for f in label_dir.glob("*.txt")}

        only_img = imgs - lbls   # NEGATIVE EXAMPLES accepted
        only_lbl = lbls - imgs   # orphan labels forbidden

        if only_img:
            print(f"[WARN] {len(only_img)} images have no labels (background images).")
        if only_lbl:
            raise ValueError(f"[ERROR] {len(only_lbl)} orphan labels detected. Fix dataset.")

    validate_yolo_pairs(train_images, train_labels)
    validate_yolo_pairs(val_images, val_labels)

    # Detect raw class IDs
    raw_class_ids = set()
    for txt_file in train_labels.glob("*.txt"):
        with open(txt_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 1 and parts[0].isdigit():
                    raw_class_ids.add(int(parts[0]))

    raw_class_ids = sorted(list(raw_class_ids))
    if not raw_class_ids:
        raise ValueError("[ERROR] No class IDs found in YOLO labels.")

    # SAFEST rule:
    # num_classes = max_class_id + 1  (not count of detected classes)
    max_id = max(raw_class_ids)
    CONFIG.num_classes = max(max_id + 1, len(raw_class_ids))

    # Default names if not manually provided
    if not CONFIG.class_names:
        CONFIG.class_names = [f"class_{i}" for i in range(CONFIG.num_classes)]

    CONFIG.include_background = False

    print(f"[INFO] YOLO class IDs detected: {raw_class_ids}")
    print(f"[INFO] Final num_classes = {CONFIG.num_classes}")
    print(f"[INFO] Class names: {CONFIG.class_names}")

    # YOLO data.yaml (relative paths)
    data_yaml = root / "data.yaml"
    ycfg = {
        "train": "train/images",
        "val": "val/images",
        "nc": CONFIG.num_classes,
        "names": CONFIG.class_names
    }

    if not data_yaml.exists():
        with open(data_yaml, "w") as f:
            yaml.dump(ycfg, f)
        print(f"[INFO] Created YOLO data.yaml at: {data_yaml}")
    else:
        with open(data_yaml, "r") as f:
            existing = yaml.safe_load(f)
        updated = False
        for k in ["nc", "names"]:
            if existing.get(k) != ycfg[k]:
                existing[k] = ycfg[k]
                updated = True
        if updated:
            with open(data_yaml, "w") as f:
                yaml.dump(existing, f)
            print("[INFO] Updated data.yaml to match dataset.")
        else:
            print("[INFO] Valid data.yaml found")

# -----------------------------
# COCO DATASET VALIDATION
# -----------------------------
elif CONFIG.data_format == "coco":
    ann_train = root / "annotations" / "instances_train.json"
    ann_val   = root / "annotations" / "instances_val.json"

    for p in [ann_train, ann_val]:
        if not p.exists():
            raise FileNotFoundError(f"[ERROR] Missing annotation: {p}")

    def validate_coco_images(json_path, image_dir):
        with open(json_path) as f:
            data = json.load(f)
        file_names = {img["file_name"] for img in data["images"]}
        disk_files = {p.name for p in image_dir.glob("*") if p.suffix.lower() in [".jpg", ".png", ".jpeg"]}
        missing = file_names - disk_files
        if missing:
            raise ValueError(f"[ERROR] {len(missing)} missing image files in {json_path.name}")

    validate_coco_images(ann_train, train_images)
    validate_coco_images(ann_val, val_images)

    # Load categories
    with open(ann_train) as f:
        data = json.load(f)

    categories = data["categories"]
    raw_ids = [c["id"] for c in categories]

    CONFIG.class_names = [c["name"] for c in categories]

    # CREATE MAPPING
    # raw COCO ID → model index (0...N-1)
    id_to_idx = {cid: idx for idx, cid in enumerate(sorted(raw_ids))}
    idx_to_id = {idx: cid for cid, idx in id_to_idx.items()}

    CONFIG.coco_id_to_idx = id_to_idx
    CONFIG.idx_to_coco_id = idx_to_id

    # TorchVision & DETR require background class index = 0
    if CONFIG.include_background:
        CONFIG.num_classes = len(categories) + 1
    else:
        CONFIG.num_classes = len(categories)

    print(f"[INFO] COCO raw IDs: {raw_ids}")
    print(f"[INFO] ID → IDX map: {CONFIG.coco_id_to_idx}")
    print(f"[INFO] Final num_classes = {CONFIG.num_classes}")

else:
    raise ValueError("Invalid CONFIG.data_format")

print("[INFO] Dataset integrity + class discovery complete.")


## Data Augmentation

In [ ]:
# 5. CELL — Data Augmentation Factory

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

if CONFIG.model_type == "yolo":
    # Ultralytics YOLO handles all augmentations internally.
    # We disable Albumentations to prevent double transformations.
    train_transform = None
    val_transform = None
    bbox_params = None
    print("[INFO] YOLO selected: Albumentations disabled entirely.")

else:
    # TorchVision / DETR augmentation setup
    bbox_params = A.BboxParams(
        format="pascal_voc",               # Dataset loader will convert for us
        label_fields=["class_labels"],
        min_area=1.0,                      # prevent degenerate bbox
        min_visibility=0.1                 # filter tiny fragments
    )

    train_transform = A.Compose([
        A.LongestMaxSize(max_size=CONFIG.img_size),                            # preserve AR
        A.PadIfNeeded(CONFIG.img_size, CONFIG.img_size, border_mode=cv2.BORDER_CONSTANT),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
        A.RandomBrightnessContrast(p=0.4),
        A.ColorJitter(p=0.3),
        A.MotionBlur(p=0.1),
        A.CLAHE(p=0.1),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ], bbox_params=bbox_params)

    val_transform = A.Compose([
        A.LongestMaxSize(max_size=CONFIG.img_size),
        A.PadIfNeeded(CONFIG.img_size, CONFIG.img_size, border_mode=cv2.BORDER_CONSTANT),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ], bbox_params=bbox_params)

print("[INFO] Augmentation setup complete for model:", CONFIG.model_type)


## Dataset Class + DataLoader Wrapper

In [ ]:
# 6. CELL — Dataset Class + DataLoader Wrapper

from torch.utils.data import Dataset, DataLoader
import json
import cv2
import torch
import numpy as np

class CustomDetectionDataset(Dataset):
    def __init__(self, root, split, transform=None):
        self.root = Path(root)
        self.split = split
        self.transform = transform

        self.img_dir = self.root / "images" / split
        self.lbl_dir = self.root / "labels" / split if CONFIG.data_format == "yolo" else None

        self.items = sorted([f for f in self.img_dir.glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])

        if CONFIG.data_format == "coco":
            ann_path = self.root / "annotations" / f"instances_{split}.json"
            with open(ann_path, "r") as f:
                self.coco_data = json.load(f)

            # Build filename→id lookup (O(1) lookup)
            self.filename_to_id = {img["file_name"]: img["id"] for img in self.coco_data["images"]}

            # Group annotations by image_id (indexing)
            self.ann_index = {}
            for ann in self.coco_data["annotations"]:
                img_id = ann["image_id"]
                self.ann_index.setdefault(img_id, []).append(ann)

    def __len__(self):
        return len(self.items)

    def load_yolo_labels(self, label_path, img_w, img_h):
        boxes = []
        labels = []
        with open(label_path, "r") as f:
            for ln in f.readlines():
                parts = ln.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:])
                xmin = (xc - w/2) * img_w
                xmax = (xc + w/2) * img_w
                ymin = (yc - h/2) * img_h
                ymax = (yc + h/2) * img_h
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(cls)
        return boxes, labels

    def load_coco_labels(self, filename):
        img_id = self.filename_to_id[filename]
        anns = self.ann_index.get(img_id, [])
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            xmin = x
            ymin = y
            xmax = x + w
            ymax = y + h
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(ann["category_id"])
        return boxes, labels

    def __getitem__(self, idx):
        img_path = self.items[idx]
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        if CONFIG.data_format == "yolo":
            lbl_path = self.lbl_dir / (img_path.stem + ".txt")
            boxes, labels = self.load_yolo_labels(lbl_path, w, h)
        else:
            boxes, labels = self.load_coco_labels(img_path.name)

        # Background offset for non-YOLO
        if CONFIG.model_type != "yolo" and CONFIG.include_background:
            labels = [l + 1 for l in labels]

        # Albumentations transform (if any)
        if self.transform:
            transformed = self.transform(image=img, bboxes=boxes, class_labels=labels)
            img = transformed["image"]
            boxes = transformed["bboxes"]
            labels = transformed["class_labels"]

        # Empty bbox case safety
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            # OPTIMIZED — no-copy tensor conversion
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        return img, target


def collate_fn(batch):
    imgs, targets = list(zip(*batch))
    return list(imgs), list(targets)


if CONFIG.model_type == "yolo":
    train_loader = None
    val_loader = None
    print("[INFO] YOLO selected: Ultralytics dataloader will be used.")
else:
    train_set = CustomDetectionDataset(CONFIG.dataset_root, "train", transform=train_transform)
    val_set   = CustomDetectionDataset(CONFIG.dataset_root, "val",   transform=val_transform)

    train_loader = DataLoader(
        train_set,
        batch_size=CONFIG.batch_size,
        shuffle=True,
        num_workers=CONFIG.num_workers,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_set,
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        collate_fn=collate_fn
    )

    print(f"[INFO] DataLoaders initialized: Train={len(train_set)} Val={len(val_set)}")


## Training Engine 

In [ ]:
# 8. CELL — Training Engine (Final Version with History Return)

import torch
from torch.optim import AdamW, SGD
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import time
import os
from pathlib import Path

def train_yolo():
    data_yaml = Path(CONFIG.dataset_root) / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError("[ERROR] YOLO requires data.yaml in dataset root.")

    results = model.train(
        data=str(data_yaml),
        epochs=CONFIG.epochs,
        imgsz=CONFIG.img_size,
        batch=CONFIG.batch_size,
        lr0=CONFIG.learning_rate,
        project=CONFIG.base_out_dir,
        name=CONFIG.run_name,
        save=True,
        device=device.index if device.type == "cuda" else "cpu",
        amp=CONFIG.amp
    )

    return {
        "train_loss": [],
        "val_loss": [],
        "map50": [],
        "map5095": []
    }


def train_torchvision_model():
    if CONFIG.optimizer.lower() == "adamw":
        optimizer = AdamW(model.parameters(), lr=CONFIG.learning_rate)
    else:
        optimizer = SGD(model.parameters(), lr=CONFIG.learning_rate, momentum=0.9)

    history = {
        "train_loss": [],
        "val_loss": [],
        "map50": [],
        "map5095": []
    }

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG.epochs) \
        if CONFIG.scheduler == "cosine" else None

    scaler = GradScaler(enabled=CONFIG.amp)
    best_val_loss = float("inf")

    for epoch in range(CONFIG.epochs):
        model.train()
        epoch_loss = 0.0
        start = time.time()

        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG.epochs}"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=CONFIG.amp):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values()) if isinstance(loss_dict, dict) else loss_dict

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        print(f"[INFO] Epoch {epoch+1}: loss={avg_loss:.4f}")

        if scheduler: scheduler.step()

        model.train()
        with torch.no_grad():
            val_loss = 0
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                with autocast(enabled=CONFIG.amp):
                    loss_dict = model(images, targets)
                    loss = sum(loss_dict.values()) if isinstance(loss_dict, dict) else loss_dict

                val_loss += loss.item()

        val_loss /= len(val_loader)
        print(f"[INFO] Validation loss: {val_loss:.4f}")

        history["train_loss"].append(avg_loss)
        history["val_loss"].append(val_loss)

        ckpt_path = os.path.join(CONFIG.ckpt_dir, f"epoch_{epoch+1}.pth")
        torch.save(model.state_dict(), ckpt_path)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(CONFIG.ckpt_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)
            print(f"[INFO] Best updated: {best_path}")

    return history


In [ ]:
# 8B. CELL — Training Trigger

if CONFIG.model_type == "yolo":
    print("[INFO] Starting YOLO training...")
    train_yolo()
else:
    print(f"[INFO] Starting PyTorch training for {CONFIG.model_type}...")
    train_torchvision_model()


## Evaluation

In [ ]:
# 9. CELL — Evaluation (YOLO + TorchVision/DETR) with mAP logging

import json
from pathlib import Path
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate_yolo(history):
    print("[INFO] YOLO evaluation...")

    data_yaml = Path(CONFIG.dataset_root) / "data.yaml"
    results = model.val(
        data=str(data_yaml),
        imgsz=CONFIG.img_size,
        batch=CONFIG.batch_size,
        device=device.index if device.type == "cuda" else "cpu"
    )

    try:
        history["map50"].append(results.metrics.box.map50)
        history["map5095"].append(results.metrics.box.map)
        print(f"[INFO] YOLO AP50={results.metrics.box.map50:.4f}, AP5095={results.metrics.box.map:.4f}")
    except:
        print("[WARN] YOLO metrics missing.")

    return history


def evaluate_pytorch(history):
    print("[INFO] COCO evaluation...")

    ann_path = Path(CONFIG.dataset_root) / "annotations" / "instances_val.json"
    coco_gt = COCO(str(ann_path))

    predictions = []
    model.eval()

    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc="Evaluating"):
            images = [img.to(device) for img in images]
            outputs = model(images)

            for out, tgt in zip(outputs, targets):
                img_id = int(tgt["image_id"].item())
                H, W = tgt["orig_size"].tolist()

                if CONFIG.model_type == "fasterrcnn":
                    boxes = out["boxes"].cpu().tolist()
                    scores = out["scores"].cpu().tolist()
                    labels = out["labels"].cpu().tolist()
                else:
                    logits = out["logits"]
                    probs = logits.softmax(-1)[..., :-1]
                    scores, labels = probs.max(-1)
                    scores = scores.cpu().tolist()
                    labels = labels.cpu().tolist()

                    boxes_norm = out["boxes"].cpu().tolist()
                    boxes = []
                    for cx, cy, w, h in boxes_norm:
                        x1 = (cx - w/2) * W
                        y1 = (cy - h/2) * H
                        x2 = (cx + w/2) * W
                        y2 = (cy + h/2) * H
                        boxes.append([x1, y1, x2, y2])

                for box, score, label in zip(boxes, scores, labels):
                    x1, y1, x2, y2 = box
                    coco_cat = CONFIG.coco_ids[label]
                    predictions.append({
                        "image_id": img_id,
                        "category_id": coco_cat,
                        "bbox": [x1, y1, x2-x1, y2-y1],
                        "score": float(score)
                    })

    if len(predictions) == 0:
        print("[WARN] No preds found.")
        return history

    pred_file = Path(CONFIG.ckpt_dir) / "coco_preds.json"
    with open(pred_file, "w") as f:
        json.dump(predictions, f)

    coco_dt = coco_gt.loadRes(str(pred_file))
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    history["map5095"].append(coco_eval.stats[0])
    history["map50"].append(coco_eval.stats[1])
    return history


if CONFIG.model_type == "yolo":
    history = evaluate_yolo(history)
else:
    history = evaluate_pytorch(history)

print("[INFO] Evaluation done.")


## Training Curves Visualization

In [ ]:
# 10. CELL — Training Curve Visualization (Loss + mAP)

import matplotlib.pyplot as plt

# Detect how many epochs we actually logged (YOLO returns empty lists)
num_epochs = len(history["train_loss"])
epochs = range(1, num_epochs + 1)

if num_epochs == 0:
    print("[WARN] No loss history found (probably YOLO). Skipping loss plot.")
else:
    # LOSS CURVE
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss", color="blue")
    plt.plot(epochs, history["val_loss"], label="Validation Loss", color="orange")
    plt.title("Training Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

# MAP CURVE (only if we have values)
if any(history["map50"]) or any(history["map5095"]):
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(history["map50"]) + 1), history["map50"], label="mAP@50", color="green")
    plt.plot(range(1, len(history["map5095"]) + 1), history["map5095"], label="mAP@[.5:.95]", color="red")
    plt.title("mAP Curve")
    plt.xlabel("Epoch")
    plt.ylabel("mAP")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("[WARN] No mAP history recorded yet. Skipping mAP plot.")

print("[INFO] Training visualizations complete.")
